# Train REINFORCE on continuous actions

REINFORCE learns a stochastic policy from complete-episode returns by minimizing

$$\mathcal L_\pi=-\frac{1}{N}\sum_t\log\pi_\theta(a_t\mid s_t)\left(G_t-V_\phi(s_t)\right).$$

Here $N$ is the number of sampled steps, $G_t$ the discounted return, $V_\phi$ a learned baseline, and $\pi_\theta$ the policy. This notebook trains it on `Pendulum-v1` with a squashed diagonal-Gaussian policy for bounded continuous actions. We use lower gravity (`g=1.0`) so the compact implementation learns a strong policy within a short demonstration.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from aprenderl import REINFORCE, REINFORCEConfig
from aprenderl.utils import evaluate_policy

ENV_ID = "Pendulum-v1"
ENV_KWARGS = {"g": 1.0}

In [ ]:
env = gym.make(ENV_ID, **ENV_KWARGS)
config = REINFORCEConfig(
    learning_rate=1e-2,
    value_learning_rate=1e-3,
    gamma=0.99,
    episodes_per_update=5,
    use_baseline=True,
    normalize_returns=True,
    seed=7,
)

agent = REINFORCE(env, config=config, device="cpu")
agent.learn(total_timesteps=50_000)
env.close()

In [ ]:
returns = np.asarray(agent.episode_returns)
window = min(10, len(returns))
moving_average = np.convolve(returns, np.ones(window) / window, mode="valid")

plt.figure(figsize=(8, 4))
plt.plot(returns, alpha=0.35, label="Episode return")
plt.plot(
    np.arange(window - 1, len(returns)),
    moving_average,
    label=f"{window}-episode average",
)
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title(f"REINFORCE training on {ENV_ID}")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## Watch the trained policy

This opens a window and runs 5 episodes using deterministic actions.

In [ ]:
evaluation_env = gym.make(
    ENV_ID, render_mode="human", **ENV_KWARGS
)
try:
    result = evaluate_policy(
        agent, evaluation_env, episodes=5, deterministic=True, seed=1_000
    )
finally:
    evaluation_env.close()

print("Episode returns:", result.returns)
print(f"Mean return: {result.mean_return:.1f} +/- {result.return_std:.1f}")